# ROCK N ROLL — Semantic Retrieval + Hybrid Ranking

This notebook builds the dataset-only ML pipeline for the hackathon.

### Datasets used
- `Analytics Jobs(2).csv` — primary job/search dataset
- `DataScience Jobs(2).csv` — demand and salary intelligence
- `JDS Skill Traits(2).xlsx` — technical skills → salary-hike outcome
- `SDS Personality Traits(2).xlsx` — personality → success outcome

### Pipeline
1. Load and audit raw data
2. Clean and normalize all four datasets
3. Engineer job, skill, experience, and salary features
4. Train lightweight career-outcome models for JDS/SDS
5. Build semantic embeddings for job records
6. Build a FAISS vector index
7. Implement semantic retrieval
8. Implement hybrid ranking using semantic similarity + skill + experience + role compatibility
9. Save processed data, models, embeddings, and index for FastAPI integration

> Important: there is no historical `job_id → candidate_id → hired` label in the supplied datasets, so this notebook does **not** train a supervised Two-Tower hiring model or fabricate hiring labels.


In [1]:


import os
import re
import json
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score

from sentence_transformers import SentenceTransformer
import faiss

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

DATA_DIR = Path(".")
PROCESSED_DIR = DATA_DIR / "processed"
ARTIFACT_DIR = DATA_DIR / "artifacts"

PROCESSED_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Ready.")



Ready.


In [2]:
ANALYTICS_FILE = DATA_DIR / "Analytics Jobs(2).csv"
DS_JOBS_FILE = DATA_DIR / "DataScience Jobs(2).csv"
JDS_FILE = DATA_DIR / "JDS Skill Traits(2).xlsx"
SDS_FILE = DATA_DIR / "SDS Personality Traits(2).xlsx"

for f in [ANALYTICS_FILE, DS_JOBS_FILE, JDS_FILE, SDS_FILE]:
    print(f"{f}: {'FOUND' if f.exists() else 'MISSING'}")


Analytics Jobs(2).csv: MISSING
DataScience Jobs(2).csv: MISSING
JDS Skill Traits(2).xlsx: MISSING
SDS Personality Traits(2).xlsx: MISSING


In [ ]:

analytics = pd.read_csv(r"D:\Mota\rock-n-roll-discover-server\data\Analytics Jobs.csv")
ds_jobs = pd.read_csv(r"D:\Mota\rock-n-roll-discover-server\data\DataScience Jobs.csv")
jds = pd.read_excel(r"D:\Mota\rock-n-roll-discover-server\data\JDS Skill Traits.xlsx")
sds = pd.read_excel(r"D:\Mota\rock-n-roll-discover-server\data\SDS Personality Traits.xlsx")

print("Analytics Jobs:", analytics.shape)
print("DataScience Jobs:", ds_jobs.shape)
print("JDS Skill Traits:", jds.shape)
print("SDS Personality Traits:", sds.shape)


Analytics Jobs: (15841, 8)
DataScience Jobs: (1602, 8)
JDS Skill Traits: (139, 7)
SDS Personality Traits: (161, 7)


In [5]:

def audit(df, name):
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nDuplicate rows:", df.duplicated().sum())
    display(df.head(3))

audit(analytics, "Analytics Jobs")
audit(ds_jobs, "DataScience Jobs")
audit(jds, "JDS Skill Traits")
audit(sds, "SDS Personality Traits")



===== Analytics Jobs =====
Shape: (15841, 8)

Columns:
['s_no', 'experience', 'job_description', 'job_desig', 'job_type', 'key_skills', 'location', 'salary']

Missing values:
s_no                   0
experience             0
job_description     3508
job_desig              0
job_type           12011
key_skills             1
location               0
salary                 0
dtype: int64

Duplicate rows: 0


,s_no,experience,job_description,job_desig,job_type,key_skills,location,salary
0,1,6-10 yrs,Must be knowledgeable and experienced in relat...,Oracle EBS Technical Consultant,NaN,"Oracle SQL, PLSQL, PL, Oracle Forms, Oracle Re...","Bengaluru, Chennai",6to10
1,2,8-12 yrs,- Experience in building scalable and highly a...,Staff Software Engineer - Object Oriented Anal...,NaN,"Javascript, HTML, JQuery, Web Development, Pla...",Bengaluru,10to15
2,3,3-8 yrs,NaN,Servicenow Developer,NaN,"SNOW, Servicenow, Servicenow Developer","Hyderabad, Pune",0to3



===== DataScience Jobs =====
Shape: (1602, 8)

Columns:
['reference_no', 'company_name', 'job_title', 'min_experience', 'avg_salary', 'min_salary', 'max_salary', 'num_of_jobs']

Missing values:
reference_no      0
company_name      0
job_title         0
min_experience    0
avg_salary        0
min_salary        0
max_salary        0
num_of_jobs       0
dtype: int64

Duplicate rows: 0


,reference_no,company_name,job_title,min_experience,avg_salary,min_salary,max_salary,num_of_jobs
0,7834,TCS,Data Scientist,2,7.8L,4.5L,16.0L,841
1,7862,Accenture,Data Scientist,2,12.8L,5.8L,23.0L,501
2,5925,IBM,Data Scientist,2,13.4L,5.3L,25.0L,394



===== JDS Skill Traits =====
Shape: (139, 7)

Columns:
['id', 'big_data_skills', 'maths-stats_skills', 'coding_skills', 'ai_and_ml_skills', 'dashboard_and_storytelling_skills', 'salary_hike_high_or_low']

Missing values:
id                                   0
big_data_skills                      0
maths-stats_skills                   0
coding_skills                        0
ai_and_ml_skills                     0
dashboard_and_storytelling_skills    0
salary_hike_high_or_low              0
dtype: int64

Duplicate rows: 0


,id,big_data_skills,maths-stats_skills,coding_skills,ai_and_ml_skills,dashboard_and_storytelling_skills,salary_hike_high_or_low
0,2809,3.6,4.0,4.5,4.8,5.0,1
1,2231,5.0,3.8,3.3,5.0,4.4,0
2,3522,4.1,3.8,4.4,4.2,5.0,1



===== SDS Personality Traits =====
Shape: (161, 7)

Columns:
['id', 'neuroticism', ' extraversion', 'openness_to_experience', 'agreeableness', 'conscientiousness', 'success_ classification_ high_low']

Missing values:
id                                   0
neuroticism                          0
 extraversion                        0
openness_to_experience               0
agreeableness                        0
conscientiousness                    0
success_ classification_ high_low    0
dtype: int64

Duplicate rows: 0


,id,neuroticism,extraversion,openness_to_experience,agreeableness,conscientiousness,success_ classification_ high_low
0,8120,33,34,39,45,51,1
1,8951,50,36,31,59,39,0
2,8656,38,35,42,39,49,1


## 4. Cleaning utilities

The cleaning rules are deliberately conservative:
- IDs are retained for traceability but excluded from ML features.
- Missing job descriptions are reconstructed from available job title, skills, and experience.
- Missing job type is kept as `unknown` rather than guessed.
- Experience and salary ranges are converted into numeric features.
- Skill strings are normalized without aggressively merging unrelated technologies.


In [6]:
# 5. General cleaning helpers
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\\n", " ").replace("\\r", " ")
    x = re.sub(r"\\s+", " ", x).strip()
    return x

def normalize_skill_string(x):
    x = clean_text(x).lower()
    if not x:
        return ""
    # Normalize common separators to commas.
    x = re.sub(r"[|;/]+", ",", x)
    x = re.sub(r"\\s*,\\s*", ",", x)
    parts = [p.strip() for p in x.split(",") if p.strip()]
    # Preserve order while removing exact duplicates.
    return ", ".join(dict.fromkeys(parts))

def parse_experience(x):
    s = clean_text(x).lower()
    nums = [float(n) for n in re.findall(r"\\d+(?:\\.\\d+)?", s)]
    if not nums:
        return pd.Series({"experience_min": np.nan,
                          "experience_max": np.nan,
                          "experience_mid": np.nan})
    if len(nums) == 1:
        lo = hi = nums[0]
    else:
        lo, hi = min(nums[0], nums[1]), max(nums[0], nums[1])
    return pd.Series({"experience_min": lo,
                      "experience_max": hi,
                      "experience_mid": (lo + hi) / 2})

def parse_salary_range(x):
    s = clean_text(x).lower()
    nums = [float(n) for n in re.findall(r"\\d+(?:\\.\\d+)?", s)]
    if not nums:
        return pd.Series({"salary_min": np.nan,
                          "salary_max": np.nan,
                          "salary_mid": np.nan})
    if len(nums) == 1:
        lo = hi = nums[0]
    else:
        lo, hi = min(nums[0], nums[1]), max(nums[0], nums[1])
    return pd.Series({"salary_min": lo,
                      "salary_max": hi,
                      "salary_mid": (lo + hi) / 2})


In [7]:
# 6. Clean Analytics Jobs — primary semantic retrieval dataset
a = analytics.copy()

# Standardize column names
a.columns = [c.strip().lower().replace(" ", "_") for c in a.columns]

# Text fields
for col in ["job_description", "job_desig", "job_type", "key_skills", "location", "salary", "experience"]:
    if col in a.columns:
        a[col] = a[col].map(clean_text)

# Missing job description fallback
a["key_skills_normalized"] = a["key_skills"].map(normalize_skill_string)

a["job_text"] = (
    "Role: " + a["job_desig"] +
    ". Required skills: " + a["key_skills_normalized"] +
    ". Experience: " + a["experience"] +
    ". Job type: " + a["job_type"].replace("", "unknown") +
    ". Location: " + a["location"]
)

has_desc = a["job_description"].str.len() > 0
a.loc[has_desc, "job_text"] = (
    "Role: " + a.loc[has_desc, "job_desig"] +
    ". Description: " + a.loc[has_desc, "job_description"] +
    ". Required skills: " + a.loc[has_desc, "key_skills_normalized"] +
    ". Experience: " + a.loc[has_desc, "experience"]
)

exp_features = a["experience"].apply(parse_experience)
a = pd.concat([a, exp_features], axis=1)

salary_features = a["salary"].apply(parse_salary_range)
a = pd.concat([a, salary_features], axis=1)

# Conservative missing-value handling for structured features.
for col in ["experience_min", "experience_max", "experience_mid",
            "salary_min", "salary_max", "salary_mid"]:
    a[col] = pd.to_numeric(a[col], errors="coerce")

print("Cleaned Analytics Jobs:", a.shape)
print("Missing job descriptions after fallback:", (a["job_text"].str.len() == 0).sum())
display(a[["job_desig", "job_text", "key_skills_normalized",
           "experience_min", "experience_max", "salary_min", "salary_max"]].head(5))


Cleaned Analytics Jobs: (15841, 16)
Missing job descriptions after fallback: 0


,job_desig,job_text,key_skills_normalized,experience_min,experience_max,salary_min,salary_max
0,Oracle EBS Technical Consultant,Role: Oracle EBS Technical Consultant. Descrip...,"oracle sql, plsql, pl, oracle forms, oracle re...",NaN,NaN,NaN,NaN
1,Staff Software Engineer - Object Oriented Anal...,Role: Staff Software Engineer - Object Oriente...,"javascript, html, jquery, web development, pla...",NaN,NaN,NaN,NaN
2,Servicenow Developer,Role: Servicenow Developer. Required skills: s...,"snow, servicenow, servicenow developer",NaN,NaN,NaN,NaN
3,"Software Development Engineering, Analyst","Role: Software Development Engineering, Analys...","qtp, flex, lan, application support, performan...",NaN,NaN,NaN,NaN
4,NVH Engineer - Nastran/ Ls-dyna,Role: NVH Engineer - Nastran/ Ls-dyna. Descrip...,"hypermesh, ls - dyna, nastran, abaqus, fatigue...",NaN,NaN,NaN,NaN


In [8]:
# 7. Remove exact duplicate job records
before = len(a)
a = a.drop_duplicates().reset_index(drop=True)
print("Removed exact duplicate rows:", before - len(a))

# Save cleaned primary dataset
a.to_csv(PROCESSED_DIR / "analytics_jobs_clean.csv", index=False)
print("Saved:", PROCESSED_DIR / "analytics_jobs_clean.csv")


Removed exact duplicate rows: 0
Saved: processed\analytics_jobs_clean.csv


In [9]:
# 8. Clean DataScience Jobs — workforce planning / market intelligence
d = ds_jobs.copy()
d.columns = [c.strip().lower().replace(" ", "_") for c in d.columns]

for col in d.columns:
    if d[col].dtype == "object":
        d[col] = d[col].map(clean_text)

# Numeric conversion
for col in ["min_experience", "avg_salary", "min_salary", "max_salary", "num_of_jobs"]:
    if col in d.columns:
        d[col] = pd.to_numeric(d[col], errors="coerce")

d = d.drop_duplicates().reset_index(drop=True)

d.to_csv(PROCESSED_DIR / "datascience_jobs_clean.csv", index=False)

print("Cleaned DataScience Jobs:", d.shape)
display(d.head())


Cleaned DataScience Jobs: (1602, 8)


,reference_no,company_name,job_title,min_experience,avg_salary,min_salary,max_salary,num_of_jobs
0,7834,TCS,Data Scientist,2,NaN,NaN,NaN,841
1,7862,Accenture,Data Scientist,2,NaN,NaN,NaN,501
2,5925,IBM,Data Scientist,2,NaN,NaN,NaN,394
3,7578,Cognizant,Data Scientist,2,NaN,NaN,NaN,318
4,7177,Capgemini,Data Scientist,2,NaN,NaN,NaN,300


In [10]:
# 9. Clean JDS Skill Traits
j = jds.copy()
j.columns = [c.strip().lower().replace(" ", "_") for c in j.columns]

# Keep ID only for traceability, never as a model feature.
skill_cols = [
    "big_data_skills",
    "maths-stats_skills",
    "coding_skills",
    "ai_and_ml_skills",
    "dashboard_and_storytelling_skills",
]
target_jds = "salary_hike_high_or_low"

for col in skill_cols + [target_jds]:
    j[col] = pd.to_numeric(j[col], errors="coerce")

j = j.dropna(subset=skill_cols + [target_jds]).drop_duplicates().reset_index(drop=True)
j[target_jds] = j[target_jds].astype(int)

j.to_csv(PROCESSED_DIR / "jds_skill_traits_clean.csv", index=False)

print("Cleaned JDS:", j.shape)
print(j[target_jds].value_counts(normalize=True))
display(j.head())


Cleaned JDS: (139, 7)
salary_hike_high_or_low
1    0.52518
0    0.47482
Name: proportion, dtype: float64


,id,big_data_skills,maths-stats_skills,coding_skills,ai_and_ml_skills,dashboard_and_storytelling_skills,salary_hike_high_or_low
0,2809,3.6,4.0,4.5,4.8,5.0,1
1,2231,5.0,3.8,3.3,5.0,4.4,0
2,3522,4.1,3.8,4.4,4.2,5.0,1
3,3291,4.4,3.0,3.3,4.6,2.3,0
4,3424,2.3,2.3,2.6,3.3,3.0,0


In [13]:
# 10. Clean SDS Personality Traits

s = sds.copy()

# -----------------------------
# Clean column names robustly
# -----------------------------
s.columns = (
    s.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[\s\-]+", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)

print("Columns after cleaning:")
print(s.columns.tolist())


# -----------------------------
# Define features and target
# -----------------------------
personality_cols = [
    "neuroticism",
    "extraversion",
    "openness_to_experience",
    "agreeableness",
    "conscientiousness",
]

target_sds = "success_classification_high_low"


# -----------------------------
# Check required columns
# -----------------------------
required_cols = personality_cols + [target_sds]

missing_cols = [
    col for col in required_cols
    if col not in s.columns
]

if missing_cols:
    raise KeyError(
        f"Missing SDS columns: {missing_cols}\n"
        f"Available columns: {s.columns.tolist()}"
    )


# -----------------------------
# Convert features to numeric
# -----------------------------
for col in personality_cols:
    s[col] = pd.to_numeric(
        s[col],
        errors="coerce"
    )

s[target_sds] = pd.to_numeric(
    s[target_sds],
    errors="coerce"
)


# -----------------------------
# Remove invalid rows
# -----------------------------
before = len(s)

s = (
    s
    .dropna(subset=required_cols)
    .drop_duplicates()
    .reset_index(drop=True)
)

after = len(s)

print(f"Rows removed during cleaning: {before - after}")


# -----------------------------
# Ensure target is integer
# -----------------------------
s[target_sds] = s[target_sds].astype(int)


# -----------------------------
# Validate target values
# -----------------------------
print("\nTarget values:")
print(s[target_sds].value_counts())

print("\nTarget distribution:")
print(s[target_sds].value_counts(normalize=True))


# -----------------------------
# Display cleaned dataset
# -----------------------------
print("\nCleaned SDS shape:", s.shape)

display(s.head())


# -----------------------------
# Save cleaned dataset
# -----------------------------
output_path = PROCESSED_DIR / "sds_personality_traits_clean.csv"

s.to_csv(
    output_path,
    index=False
)

print(f"\nSaved cleaned SDS dataset to: {output_path}")

Columns after cleaning:
['id', 'neuroticism', 'extraversion', 'openness_to_experience', 'agreeableness', 'conscientiousness', 'success_classification_high_low']
Rows removed during cleaning: 0

Target values:
success_classification_high_low
1    85
0    76
Name: count, dtype: int64

Target distribution:
success_classification_high_low
1    0.52795
0    0.47205
Name: proportion, dtype: float64

Cleaned SDS shape: (161, 7)


,id,neuroticism,extraversion,openness_to_experience,agreeableness,conscientiousness,success_classification_high_low
0,8120,33,34,39,45,51,1
1,8951,50,36,31,59,39,0
2,8656,38,35,42,39,49,1
3,8796,41,26,30,40,19,0
4,8470,20,17,21,31,26,0



Saved cleaned SDS dataset to: processed\sds_personality_traits_clean.csv


# 11. Train lightweight outcome models

These models are deliberately simple because JDS has only 139 rows and SDS has only 161 rows.

We do **not** use IDs as features.

The models are auxiliary signals for the GROW / career-intelligence component, not the primary semantic retrieval engine.


In [14]:
# 12. JDS — skill profile -> salary-hike outcome
X_j = j[skill_cols]
y_j = j[target_jds]

Xj_train, Xj_test, yj_train, yj_test = train_test_split(
    X_j, y_j, test_size=0.25, random_state=RANDOM_STATE, stratify=y_j
)

jds_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                               random_state=RANDOM_STATE))
])

jds_model.fit(Xj_train, yj_train)
pred_j = jds_model.predict(Xj_test)
prob_j = jds_model.predict_proba(Xj_test)[:, 1]

print("JDS Accuracy:", round(accuracy_score(yj_test, pred_j), 4))
print("JDS F1:", round(f1_score(yj_test, pred_j), 4))
if len(np.unique(yj_test)) == 2:
    print("JDS ROC-AUC:", round(roc_auc_score(yj_test, prob_j), 4))
print(classification_report(yj_test, pred_j, zero_division=0))

joblib.dump(jds_model, ARTIFACT_DIR / "jds_salary_hike_model.joblib")


JDS Accuracy: 0.8571
JDS F1: 0.8571
JDS ROC-AUC: 0.9755
              precision    recall  f1-score   support

           0       0.83      0.88      0.86        17
           1       0.88      0.83      0.86        18

    accuracy                           0.86        35
   macro avg       0.86      0.86      0.86        35
weighted avg       0.86      0.86      0.86        35



['artifacts\\jds_salary_hike_model.joblib']

In [15]:
# 13. SDS — personality profile -> success outcome
X_s = s[personality_cols]
y_s = s[target_sds]

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_s, y_s, test_size=0.25, random_state=RANDOM_STATE, stratify=y_s
)

sds_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                               random_state=RANDOM_STATE))
])

sds_model.fit(Xs_train, ys_train)
pred_s = sds_model.predict(Xs_test)
prob_s = sds_model.predict_proba(Xs_test)[:, 1]

print("SDS Accuracy:", round(accuracy_score(ys_test, pred_s), 4))
print("SDS F1:", round(f1_score(ys_test, pred_s), 4))
if len(np.unique(ys_test)) == 2:
    print("SDS ROC-AUC:", round(roc_auc_score(ys_test, prob_s), 4))
print(classification_report(ys_test, pred_s, zero_division=0))

joblib.dump(sds_model, ARTIFACT_DIR / "sds_success_model.joblib")


SDS Accuracy: 0.9268
SDS F1: 0.9333
SDS ROC-AUC: 0.9402
              precision    recall  f1-score   support

           0       0.94      0.89      0.92        19
           1       0.91      0.95      0.93        22

    accuracy                           0.93        41
   macro avg       0.93      0.92      0.93        41
weighted avg       0.93      0.93      0.93        41



['artifacts\\sds_success_model.joblib']

# 14. Semantic retrieval

We use a pretrained sentence-transformer instead of training a Two-Tower network from scratch.

The embedding text contains the job's:
- role
- description
- normalized skills
- experience

This makes semantic retrieval robust even when exact keywords differ.


In [16]:
# 15. Load embedding model
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedder.get_sentence_embedding_dimension())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/model.safetensors: HTTPSConnectionPool(host='us.aws.cdn.hf.co', port=443): Read timed out.
Trying to resume download...


model.safetensors:  58%|#####7    | 52.4M/90.9M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


In [17]:
# 16. Generate job embeddings
job_texts = a["job_text"].fillna("").tolist()

job_embeddings = embedder.encode(
    job_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

print("Embedding matrix:", job_embeddings.shape)

np.save(ARTIFACT_DIR / "analytics_job_embeddings.npy", job_embeddings)
joblib.dump(
    {"model_name": EMBEDDING_MODEL_NAME},
    ARTIFACT_DIR / "embedding_config.joblib"
)


Batches:   0%|          | 0/248 [00:00<?, ?it/s]

Embedding matrix: (15841, 384)


['artifacts\\embedding_config.joblib']

In [18]:
# 17. Build FAISS index
dimension = job_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)  # inner product = cosine similarity because vectors are normalized
index.add(job_embeddings)

faiss.write_index(index, str(ARTIFACT_DIR / "analytics_jobs.faiss"))

print("FAISS vectors:", index.ntotal)


FAISS vectors: 15841


# 18. Hybrid ranking

For a recruiter query, we calculate:

- semantic similarity
- skill overlap
- experience compatibility
- role/title compatibility

Then combine normalized signals into a final score.

The weights below are **baseline engineering weights**, not claimed as learned ground truth. They should be tuned after you create a proper validation set.


In [23]:
# 19. Hybrid matching helpers
STOPWORDS = {
    "and", "or", "the", "a", "an", "with", "for", "to", "of", "in",
    "on", "at", "from", "by", "is", "are", "as", "be"
}

def tokenize(text):
    text = clean_text(text).lower()
    tokens = re.findall(r"[a-zA-Z0-9+#.]+", text)
    return {t for t in tokens if t not in STOPWORDS}

def extract_skill_set(text):
    return set(
        x.strip().lower()
        for x in normalize_skill_string(text).split(",")
        if x.strip()
    )

def skill_overlap(query_skills, row_skills):
    q = extract_skill_set(query_skills)
    r = extract_skill_set(row_skills)
    if not q:
        return 0.0
    return len(q & r) / len(q)

def role_overlap(query_role, row_role):
    q = tokenize(query_role)
    r = tokenize(row_role)
    if not q:
        return 0.0
    return len(q & r) / len(q)

def experience_compatibility(required_experience, candidate_experience):
    req = parse_experience(required_experience)
    cand = parse_experience(candidate_experience)

    req_mid = req["experience_mid"]
    cand_mid = cand["experience_mid"]

    if pd.isna(req_mid) or pd.isna(cand_mid):
        return 0.5

    # Full score if candidate meets/exceeds the requested midpoint.
    if cand_mid >= req_mid:
        return 1.0

    # Graceful decrease when candidate is below the midpoint.
    return max(0.0, cand_mid / max(req_mid, 1.0))


In [20]:
# 20. Semantic retrieval + hybrid ranking function
def search_jobs(
    query,
    required_skills="",
    required_experience="",
    role="",
    top_k_retrieve=50,
    top_k_return=10,
    semantic_weight=0.55,
    skill_weight=0.25,
    experience_weight=0.10,
    role_weight=0.10
):
    q_embedding = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    distances, indices = index.search(q_embedding, top_k_retrieve)

    results = a.iloc[indices[0]].copy()
    results["semantic_score"] = distances[0]

    results["skill_score"] = results["key_skills"].fillna("").map(
        lambda x: skill_overlap(required_skills, x)
    )

    results["experience_score"] = results["experience"].fillna("").map(
        lambda x: experience_compatibility(required_experience, x)
    )

    results["role_score"] = results["job_desig"].fillna("").map(
        lambda x: role_overlap(role, x)
    )

    # Cosine similarity is already approximately in [-1, 1].
    # Map to [0, 1] for combining with other scores.
    results["semantic_score_norm"] = (
        results["semantic_score"].clip(-1, 1) + 1
    ) / 2

    results["hybrid_score"] = (
        semantic_weight * results["semantic_score_norm"] +
        skill_weight * results["skill_score"] +
        experience_weight * results["experience_score"] +
        role_weight * results["role_score"]
    )

    return results.sort_values(
        "hybrid_score", ascending=False
    ).head(top_k_return).reset_index(drop=True)


In [21]:
# 21. Example search
example_query = (
    "Data Scientist responsible for building machine learning models, "
    "working with Python, statistics and data analysis."
)

results = search_jobs(
    query=example_query,
    required_skills="Python, Machine Learning, Statistics",
    required_experience="2-5 yrs",
    role="Data Scientist",
    top_k_retrieve=50,
    top_k_return=10
)

display(results[[
    "job_desig",
    "experience",
    "key_skills",
    "semantic_score",
    "skill_score",
    "experience_score",
    "role_score",
    "hybrid_score"
]])


,job_desig,experience,key_skills,semantic_score,skill_score,experience_score,role_score,hybrid_score
0,Data Scientist - R/python/machine Learning,3-5 yrs,"Spark, Big Data, Machine Learning, R, Python, ...",0.702484,1.000000,0.5,1.0,0.868183
1,Data Scientist - R/python/machine Learning,1-5 yrs,"Spark, Big Data, Machine Learning, R, Python, ...",0.702426,1.000000,0.5,1.0,0.868167
2,Data Scientist,5-7 yrs,"data scientist, machine learning, python, anal...",0.736070,0.666667,0.5,1.0,0.794086
3,Data Scientist - Machine Learning,4-9 yrs,"Text Mining, R, Machine Learning, Python, Pred...",0.724973,0.666667,0.5,1.0,0.791034
4,Data Scientist - Machine Learning/predictive M...,2-5 yrs,"Machine Learning, Data Science, R, Python, SQL...",0.720931,0.666667,0.5,1.0,0.789923
5,Data Analyst | Data Scientist | Machine Learni...,0-5 yrs,"data science, machine learning, python, hadoop...",0.718444,0.666667,0.5,1.0,0.789239
6,"Data Scientist - Machine Learning, C, C++, Jav...",10-12 yrs,"Machine Learning, C, C++, Java, Python, R, Dat...",0.708861,0.666667,0.5,1.0,0.786604
7,Data Scientist - Chennai,7-12 yrs,"Data Science, Text Analytics, Machine Learning...",0.706874,0.666667,0.5,1.0,0.786057
8,Good Opportuity for Data Scientist,5-8 yrs,"R, PYTHON, Machine Learning",0.699969,0.666667,0.5,1.0,0.784158
9,Senior Data Scientist,5-10 yrs,"machine learning, data science, python, predic...",0.698400,0.666667,0.5,1.0,0.783727


# 22. Save final metadata

These files are what the later FastAPI/LangGraph service will load.

Artifacts:
- cleaned CSVs
- JDS outcome model
- SDS success model
- SentenceTransformer model name
- job embedding matrix
- FAISS index


In [22]:
# 23. Save metadata and summary
metadata = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "analytics_rows": int(len(a)),
    "datascience_rows": int(len(d)),
    "jds_rows": int(len(j)),
    "sds_rows": int(len(s)),
    "semantic_weight": 0.55,
    "skill_weight": 0.25,
    "experience_weight": 0.10,
    "role_weight": 0.10,
    "note": "Weights are baseline engineering values and should be tuned on a validation set."
}

with open(ARTIFACT_DIR / "pipeline_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))
print("\nPipeline artifacts saved to:", ARTIFACT_DIR.resolve())


{
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "analytics_rows": 15841,
  "datascience_rows": 1602,
  "jds_rows": 139,
  "sds_rows": 161,
  "semantic_weight": 0.55,
  "skill_weight": 0.25,
  "experience_weight": 0.1,
  "role_weight": 0.1,
  "note": "Weights are baseline engineering values and should be tuned on a validation set."
}

Pipeline artifacts saved to: D:\Mota\rock-n-roll-discover-server\artifacts


# 24. Next integration step

The notebook is the offline ML/data-preparation stage.

The next project layer should load:
- `analytics_jobs_clean.csv`
- `analytics_jobs.faiss`
- `analytics_job_embeddings.npy`
- `jds_salary_hike_model.joblib`
- `sds_success_model.joblib`
- `pipeline_metadata.json`

Then FastAPI + LangGraph can call the retrieval/ranking functions.

## Important limitation

The supplied datasets do not contain explicit historical job-candidate hiring/matching labels. Therefore this pipeline does **not** claim supervised hiring accuracy or train a fabricated Two-Tower matching model.


u